In [23]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [24]:
start_time = time.time()

In [25]:
bd.projects.set_current('propanol_production_monte_carlo')

In [26]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'inventories-ei310']

In [27]:
methods = [
    # ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: human health no LT', 'human health no LT'),
#  ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: ecosystem quality no LT', 'ecosystem quality no LT'),
 ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: natural resources no LT', 'natural resources no LT')
]

In [28]:
methods

[('ecoinvent-3.10',
  'ReCiPe 2016 v1.03, endpoint (H) no LT',
  'total: natural resources no LT',
  'natural resources no LT')]

In [29]:
db = bd.Database('inventories-ei310')
acts = [act for act in db if 'propanol production' in act['name'] and 'GLO' in act['location']]
len(acts)

5

In [30]:
mc_df = pd.DataFrame()

In [31]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(500):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [32]:
mc_df

,"propanol production, DAC carbon dioxide, wind hydrogen, fossil ethylene","propanol production, biogas, green ethylene","propanol production, fossil","propanol production, biogas, fossil ethylene","propanol production, DAC carbon dioxide, wind hydrogen, green ethylene"
0,0.438583,0.139977,1.158830,0.356390,0.221812
1,0.448125,0.130467,0.736720,0.364974,0.213230
2,0.419193,0.143042,0.820601,0.343820,0.218083
3,0.470002,0.185913,1.142801,0.373405,0.282200
4,0.411317,0.122260,1.065972,0.334481,0.198745
...,...,...,...,...,...
495,0.642750,0.262373,1.335118,0.519533,0.385164
496,0.482494,0.129791,0.984244,0.386690,0.225171
497,0.458672,0.152556,0.858023,0.372740,0.238123
498,0.440906,0.155995,0.613265,0.359490,0.237075


In [33]:
# mc_df.to_excel(os.path.join('..', 'results', 'human_health_monte_carlo.xlsx'))
# mc_df.to_excel(os.path.join('..', 'results', 'ecosystem_quality_monte_carlo.xlsx'))
mc_df.to_excel(os.path.join('..', 'results', 'natural_resources_monte_carlo.xlsx'))